# Multi-Hop Retrieval — Train ComplementGenerator (G+D, retrieval_v3)

**Before running:**
1. Settings → Accelerator → **T4 GPU**
2. Settings → Internet → **On**
3. Google Drive `musique_data/` folder must contain:
   - `musique_ans_v1.0_train.jsonl`
   - `musique_ans_v1.0_dev.jsonl`

**What this trains — G+D architecture (fixes v1/v2 collapse):**
- **Generator G(A,B)**: shared BERT sees A and B directly; B→A cross-attention +
  learned λ subtraction + complement gate → 128-dim edge vector
- **Discriminator D(A,e)**: reconstructs B from A + edge, with 40% context dropout
  on A so D is forced to use the edge
- **Loss**: L_rec (reconstruction) + 0.1 × InfoNCE diversity

**Critical gate:** `collapse_sim` should drop below **0.80** by epoch 2.
Both v1 and v2 were stuck at >0.95 (complement = encode(B)).

**Expected time: ~5-6 hours on T4**

**After training:** download `generator_best.pt` from Output tab → upload to Drive `musique_data/`

In [ ]:
# ── [EDIT IF NEEDED] ─────────────────────────────────────────────────
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"   # musique_data folder ID
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/retrieval_v3"
# ─────────────────────────────────────────────────────────

In [ ]:
# 1. Verify GPU — must be T4 (sm_70+), not P100 (sm_60)
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Settings → Accelerator → T4 GPU")

props   = torch.cuda.get_device_properties(0)
cc      = props.major
vram_gb = props.total_memory / 1e9

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"SM   : {cc}.{props.minor}")

if cc < 7:
    raise RuntimeError(
        f"GPU is P100 (sm_{cc}0) — not supported by PyTorch 2.x.\n"
        "FIX: Settings → Accelerator → change to T4 GPU"
    )
print("CUDA OK")

In [ ]:
# 2. Clone repo + install dependencies
import os

repo_root = f"/kaggle/working/{REPO_NAME}"
if not os.path.isdir(repo_root):
    os.system(f"git clone {REPO_URL} {repo_root}")
else:
    os.system(f"cd {repo_root} && git pull")

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())

os.system("pip install -q 'transformers>=4.35.0' 'rank_bm25>=0.2.2' gdown")
print("Dependencies ready")

In [ ]:
# 3. Download MuSiQue data from Google Drive (no prior checkpoint needed)
import os, gdown

download_dir = "/kaggle/working/drive_data"

if not os.path.isdir(download_dir):
    print("Downloading from Google Drive...")
    gdown.download_folder(
        id=DRIVE_FOLDER_ID,
        output=download_dir,
        quiet=False,
        use_cookies=False,
    )
else:
    print("Drive data already downloaded")

print("\nDownloaded files:")
for f in sorted(os.listdir(download_dir)):
    size = os.path.getsize(f"{download_dir}/{f}") / 1e6
    print(f"  {f:45s}  {size:7.1f} MB")

In [ ]:
# 4. Set up file paths — v3 uses data_loader from retrieval_v2/, so symlink data there
import os

drive   = "/kaggle/working/drive_data"
v2_data = f"/kaggle/working/{REPO_NAME}/retrieval_v2/data/musique"

os.makedirs(v2_data,              exist_ok=True)
os.makedirs(f"{WORK_DIR}/models", exist_ok=True)
os.makedirs(f"{WORK_DIR}/cache",  exist_ok=True)
os.makedirs(f"{WORK_DIR}/results",exist_ok=True)

# Symlink MuSiQue JSONL into retrieval_v2/data/musique (data_loader looks there)
for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src = f"{drive}/{fname}"
    dst = f"{v2_data}/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: OK ({os.path.getsize(src)/1e6:.0f} MB)")

print("\nAll paths ready")

In [ ]:
# 5. Smoke test — verify model builds, data loads, one epoch runs
import os
os.chdir(WORK_DIR)

print("=== Smoke test (50 examples, 1 epoch) ===")
os.system("python generator_train.py --smoke")

---
## Train ComplementGenerator — Full 3-epoch run

**Architecture:**
- Generator G(A,B): shared BERT → B→A cross-attention → λ subtraction → complement gate → 128-dim edge
- Discriminator D(A,e): edge as K2/V2 → 2-layer causal decoder reconstructs B (40% context dropout on A)
- Loss: L_rec + 0.1 × InfoNCE diversity

**Watch the `collapse_sim` value in the logs:**
- Printed every 200 steps and at end of each epoch
- Target: drops below 0.80 by epoch 2
- If stuck >0.95: collapse (same as v1/v2) — the edge is not carrying unique info

In [ ]:
# 6. Train (full 3-epoch run)
import os
os.chdir(WORK_DIR)

log_file = "/kaggle/working/generator_train.log"
os.system(f"python generator_train.py 2>&1 | tee {log_file}")
print("\nTraining complete")

In [ ]:
# 7. Verify checkpoints + copy generator_best.pt to /kaggle/working/ for download
import os, shutil

best  = f"{WORK_DIR}/models/generator_best.pt"
final = f"{WORK_DIR}/models/generator_final.pt"

for path in [best, final]:
    if os.path.exists(path):
        print(f"  {os.path.basename(path)}: {os.path.getsize(path)/1e6:.1f} MB  OK")
    else:
        print(f"  {os.path.basename(path)}: NOT FOUND — check training log")

if os.path.exists(best):
    out_path = "/kaggle/working/generator_best.pt"
    shutil.copy(best, out_path)
    print(f"\n  Copied to {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)")
    print("  → Download from the Output tab and upload to Drive musique_data/")

print("\n--- Last 30 lines of training log ---")
with open("/kaggle/working/generator_train.log") as f:
    lines = f.readlines()
print("".join(lines[-30:]))

---
## Done

`generator_best.pt` is in `/kaggle/working/` — visible in the **Output tab**.

**To save to Google Drive:**
1. Output tab → find `generator_best.pt` → download
2. Upload it to your Google Drive `musique_data/` folder

**Next steps:**
1. (Optional) Train Model 2: run `train_model2_kaggle.ipynb` after uploading generator_best.pt
2. Evaluate: run `eval_kaggle.ipynb` in retrieval_v3/

**Decision gate (from eval):** if `collapse_sim < 0.85` AND `FULL > Graph+cosine`,
the G+D architecture has broken the collapse that killed v1 and v2.